# 03 - Feature Engineering

This notebook creates rolling team-form features for each match. The key rule is that features for a match may only use games played before. This prevents data leakage, making model evaluation more realistic.

In [2]:
from pathlib import Path
from collections import defaultdict
import json

import pandas as pd
import numpy as np

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [3]:
results = pd.read_csv(PROCESSED_DIR / "clean_matches.csv")
results["date"] = pd.to_datetime(results["date"])

results = results.sort_values("date").reset_index(drop=True)

results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,total_goals,result
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,0,draw
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,6,home_win
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,3,home_win
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,4,draw
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,3,home_win


## Data Leakage Rule

For each match, the model can only use information from matches that happened earlier. If the model accidentally uses future matches, the performance numbers will look better than they really are.

To avoid this, the feature-building loop processes the data in chronological order. For each row, it calculates each team's recent form first, then updates that team's history after the match is processed.

In [7]:
def summarize_recent(history, n=10):
    recent = history[-n:]
    games = len(recent)

    if games == 0:
        return {
            "games_available": 0,
            "win_rate": np.nan,
            "draw_rate": np.nan,
            "loss_rate": np.nan,
            "avg_goals_for": np.nan,
            "avg_goals_against": np.nan,
            "avg_goal_diff": np.nan,
        }

    goals_for = np.array([match["gf"] for match in recent], dtype=float)
    goals_against = np.array([match["ga"] for match in recent], dtype=float)
    outcomes = [match["outcome"] for match in recent]

    return {
        "games_available": games,
        "win_rate": outcomes.count("win") / games,
        "draw_rate": outcomes.count("draw") / games,
        "loss_rate": outcomes.count("loss") / games,
        "avg_goals_for": goals_for.mean(),
        "avg_goals_against": goals_against.mean(),
        "avg_goal_diff": (goals_for - goals_against).mean(),
    }
